# 08 · 用 REINFORCE 学一个驾驶速度选择

这里学习的对象很窄：方向盘仍由固定几何控制器计算，策略每个 `decision_repeat=5` 的决策周期从 `{2,6} m/s` 中采样一个目标速度；这个目标只通过真实油门进入 `env.step`。所以结果应称为“固定转向之上的纵向选择”，不是完整自动驾驶策略。

本课使用模拟器特权真值（privileged truth），没有接入第二单元的带噪测量或滤波器，以单独研究策略更新。五个观测特征是车道横向误差、朝向误差、速度和两个速度误差；奖励从实际 trace 的纵向进度、横向误差和失败标志重算。4步执行队列没有放入这五个特征，因此这是部分可观测过程上的无记忆策略；REINFORCE在这个受限策略类中优化回报。

In [ ]:
from pathlib import Path
import sys
import numpy as np
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src" / "ad_tutorial").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from ad_tutorial.rl_foundations import *

from dataclasses import replace
from ad_tutorial.driving import DrivingConfig
from ad_tutorial.rl_foundations import (SpeedChoicePolicy, FixedSpeedPolicy, run_policy_episode, trace_rewards, reinforce_loss, train_reinforce, save_checkpoint, load_checkpoint)

## 1. 先看动作链，不先看分数

`observation → categorical policy → target speed → geometric throttle → run_episode → MetaDrive state`。`run_episode` 记录 issued command、delayed applied command 和下一状态。延迟 4 步且每步 0.1 秒时，前 0.4 秒执行器还在排队；策略依然每个决策周期根据当时测得状态重新采样。训练用随机 categorical policy，评测用 checkpoint reload 后的 deterministic argmax deployment；评测回报不能当作训练随机 `J` 的无偏估计。

对 `J=E[Σ_t γ^t r_t]`，REINFORCE 使用 `Σ_t γ^t ∇ log π(a_t|s_t) G_t`。这里采用有限 horizon Monte Carlo，终点 bootstrap=0；每条 episode 的损失做求和，批量时再按 episode 数平均，不按各自长度除。`dist.sample()` 加 `log_prob` 是 score-function estimator；不要把离散速度选择写成 rsample 后又混合路径梯度，那会改变估计器含义。

In [ ]:
config = DrivingConfig(seed=7, horizon=45, action_delay_steps=4, decision_repeat=5)
policy = SpeedChoicePolicy(seed=7)
episode = run_policy_episode(config, policy)
print({"steps": len(episode.rewards), "actions": episode.actions[:8],
       "return": float(episode.rewards.sum()), "metrics": episode.result.metrics})
assert len(episode.actions) == len(episode.rewards) == len(episode.log_probs)
assert np.allclose(episode.rewards, trace_rewards(episode.result.trace))

## 2. baseline 只减方差，不替换目标

下面的 running mean 只使用先前轨迹的回报。对当前动作独立的 baseline 满足 `E_a[∇logπ(a|s)b]=b∇Σ_aπ(a|s)=0`，因此不改变期望梯度；合适的 baseline 可以减小方差，任意 baseline 未必更好。用当前轨迹自己的均值/标准差处理 return 会依赖当前动作，单轨迹时可能引入偏差；本课不采用它。

### 手算 score-function 梯度

对 `p=softmax([0,0])=[.5,.5]`，选择 action 0 时
`∂log softmax_0/∂logits = [1-p0, -p1] = [.5,-.5]`。
因而单步 loss `-2 log p0` 的梯度是 `[-1,+1]`。这个梯度只来自
policy 的 log-prob；环境、reward 和 MetaDrive 没有路径梯度。

In [ ]:
import torch
from torch.distributions import Categorical

# Analytic two-action check: logits=[0,0], sampled action 0, return 2.
analytic_logits = torch.zeros(2, requires_grad=True)
analytic_log_prob = Categorical(logits=analytic_logits).log_prob(torch.tensor(0))
analytic_loss = reinforce_loss([analytic_log_prob], [2.0], gamma=1.0, baseline="none")
analytic_loss.backward()
print("analytic gradient:", analytic_logits.grad.tolist())
assert torch.allclose(analytic_logits.grad, torch.tensor([-1.0, 1.0]))

# An SGD step raises p(action=0) for this positive-return sample.
before_probability = Categorical(logits=analytic_logits.detach()).probs[0].item()
optimizer_logits = torch.optim.SGD([analytic_logits], lr=0.5)
optimizer_logits.step()
after_probability = Categorical(logits=analytic_logits.detach()).probs[0].item()
print("p(action=0) before/after:", before_probability, after_probability)
assert after_probability > before_probability

loss = reinforce_loss(episode.log_probs, episode.rewards, baseline="running_mean", baseline_value=0.0)
before = [p.detach().clone() for p in policy.parameters()]
optimizer = torch.optim.Adam(policy.parameters(), lr=0.01)
optimizer.zero_grad(); loss.backward(); optimizer.step()
print("loss", float(loss), "parameter changed", any(not torch.equal(a, b) for a, b in zip(before, policy.parameters())))

## 3. 小规模训练与公平评测

下面的 notebook 示例只跑一个独立模型，便于逐 cell 修改；CLI 会从 `--seed` 派生 3 个独立模型/采样 seed，并为每个保存训练前后 checkpoint。所有模型使用相同固定地图上的匹配初始偏移，最终再用未参与选择的 seed/offset 作为 holdout。报告 discounted objective、undiscounted reward sum、失败、距离和失败原因。一次短 CPU 运行若没有改善，就保留这个结果，解释高方差、延迟和有限样本，而不是挑一条好看的轨迹。

In [ ]:
trained = train_reinforce(config, episodes=3, seeds=(7, 11, 19), checkpoint=ROOT / "artifacts" / "rl_foundations" / "lesson_policy.pt")
print(trained.history)

In [ ]:
reloaded = load_checkpoint(ROOT / "artifacts" / "rl_foundations" / "lesson_policy.pt")
a = run_policy_episode(config, reloaded)
b = run_policy_episode(config, reloaded)
reloaded_actions = a.actions
print("reloaded deterministic actions:", reloaded_actions[:10])
assert reloaded_actions == b.actions

## 4. 一个因素实验：只去掉执行延迟

使用同一个已保存 checkpoint、同一个 seed/offset、同一个 horizon，只把 `action_delay_steps` 从 4 改成 0。运行前预测：队列不再隐藏旧动作，实际 throttle 更快跟随命令，进度和失败风险可能改变，但不能把这个单因素结果解释成泛化或训练胜利。记录 issued/applied command、实际 progress、undiscounted reward、discounted deployment return `G_0` 和 failure，再解释代价。

In [ ]:
one_factor = DrivingConfig(seed=7, horizon=45, action_delay_steps=4, decision_repeat=5)
delayed = run_policy_episode(one_factor, reloaded)
no_delay = run_policy_episode(replace(one_factor, action_delay_steps=0), reloaded)
def one_factor_report(ep):
    return {
        "delay_steps": ep.result.config["action_delay_steps"],
        "longitudinal_progress_m": ep.result.trace[-1]["longitudinal_m"] - ep.result.trace[0]["before_longitudinal_m"],
        "undiscounted_reward_sum": float(ep.rewards.sum()),
        "discounted_deployment_return": float(ep.returns[0]),
        "failure": ep.result.metrics["failure"],
        "first_issued_applied": [(ep.result.trace[0]["command_throttle"], ep.result.trace[0]["applied_throttle"])],
    }
print("delay4:", one_factor_report(delayed))
print("delay0:", one_factor_report(no_delay))

## 5. 三个独立 seed 的结论表

CLI 的最终表格必须逐行保留 model seed，并报告 `G_0`、undiscounted reward、distance 和 failure 的 learned−untrained delta。跨条件和 seed 的均值是等权汇总；没有要求每个 seed 都赢。下面的 cell 读取 CLI 产物，避免在 notebook 中挑选最好的一次运行。

In [ ]:
import json
evaluation_path = ROOT / "artifacts" / "rl_foundations" / "evaluation.json"
if evaluation_path.exists():
    report = json.loads(evaluation_path.read_text(encoding="utf-8"))
    print("model_seed | mean ΔG0 | mean Δreward | mean Δdistance | mean Δfailure")
    for run in report["runs"]:
        deltas = run["paired_deltas"]["learned_minus_untrained"]
        n = len(deltas)
        mean = lambda key: sum(float(row[key]) for row in deltas) / n
        print(run["model_seed"], mean("discounted_deployment_return_delta"),
              mean("undiscounted_reward_sum_delta"),
              mean("distance_traveled_m_delta"), mean("failure_delta"))
else:
    print("先运行 scripts/run_rl_foundations.py，再读取三 seed 结论表。")

## 6. 结果解释和下一问

这次实验的因果证据是：每个 categorical 选择改变连续 throttle，throttle 经 MetaDrive 动力学改变真实速度、进度和可能的失败标志。它没有学习 steering，也没有视觉、交互交通或路线规划；真值/测量接口和单一路段限制了泛化结论。

REINFORCE 的优点是公式短、目标直接，代价是回报噪声大、样本效率低、更新对学习率和 seed 敏感。下一步问题是：如何用 value baseline、批量轨迹和 PPO 的 clipped objective 降低更新风险？先确认这里的 on-policy 数据和 reward-to-go 对齐，再进入 PPO。

选读：[Sutton & Barto 在线教材](http://incompleteideas.net/book/the-book-2nd.html)；[OpenAI Spinning Up 的 Vanilla Policy Gradient](https://spinningup.openai.com/en/latest/algorithms/vpg.html) 与 [policy optimization 推导](https://spinningup.openai.com/en/latest/spinningup/rl_intro3.html)。